# ⚡ Official OpenBMB VoxCPM2 Studio Server (BT-Dubber GPU Engine)
### ដំណើរការម៉ាស៊ីន AI Voice Cloning ល្បឿនលឿន (Turbo 2.5s) លើ Google Colab GPU T4 ឥតគិតថ្លៃ

**របៀបដំណើរការ (3 ជំហានងាយៗ):**
1. ចុច **Runtime** ➔ **Change runtime type** ➔ ជ្រើសរើស **T4 GPU**
2. ចុចប៊ូតុង **Play (▶️ Run)** លើក្រឡាកូដខាងក្រោម
3. ចម្លងយក **Public URL** (ឧ. `https://xxxx.trycloudflare.com`) មកបិទភ្ជាប់ក្នុង **BT-Dubber**

In [ ]:
# @title 🚀 1-Click Launch VoxCPM2 Studio Turbo Server on Colab GPU

import os, sys, subprocess
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print("📥 1/4 Installing Official OpenBMB VoxCPM2 & Dependencies...")
!pip install -q --no-warn-conflicts voxcpm soundfile librosa fastapi uvicorn pydantic pycloudflared torch torchaudio numpy edge-tts huggingface_hub scipy

print("🧠 2/4 Initializing VRAM-Optimized Acoustic Reference Engine on GPU...")
import glob, json, time, socket, base64, tempfile, threading, torch, re, asyncio
import numpy as np
import soundfile as sf
import librosa
import uvicorn
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

try:
    from pycloudflared import try_cloudflare
except ImportError:
    try_cloudflare = None

device = "cuda:0" if torch.cuda.is_available() else "cpu"
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print(f"🚀 Running on: {gpu_name} ({device})")

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle") else ("/content" if os.path.exists("/content") else os.getcwd())

# Broadcast Studio Mastering Chain
def studio_voice_mastering(raw_wav, sr=48000, target_lufs_rms=-18.0, pitch_shift_semitones=0.0):
    if raw_wav is None or len(raw_wav) == 0: return raw_wav
    audio = np.array(raw_wav, dtype=np.float32) - np.mean(raw_wav)
    if pitch_shift_semitones != 0.0:
        try: audio = librosa.effects.pitch_shift(audio, sr=sr, n_steps=pitch_shift_semitones)
        except Exception: pass
    try:
        trimmed, _ = librosa.effects.trim(audio, top_db=32, frame_length=1024, hop_length=256)
        if len(trimmed) > int(sr * 0.1): audio = trimmed
    except Exception: pass
    fade_len = min(int(sr * 0.03), len(audio) // 4)
    if fade_len > 0:
        audio[:fade_len] *= np.linspace(0.0, 1.0, fade_len, dtype=np.float32)
        audio[-fade_len:] *= np.linspace(1.0, 0.0, fade_len, dtype=np.float32)
    try:
        from scipy.signal import butter, sosfilt
        sos_hp = butter(2, 80.0, btype='highpass', fs=sr, output='sos')
        audio = sosfilt(sos_hp, audio)
        sos_shelf = butter(1, 3500.0, btype='highpass', fs=sr, output='sos')
        audio = audio + (0.22 * sosfilt(sos_shelf, audio))
    except Exception: pass
    drive = 1.35
    compressed = np.tanh(audio * drive) / np.tanh(drive)
    audio = (0.75 * compressed) + (0.25 * audio)
    rms = np.sqrt(np.mean(audio**2)) + 1e-7
    target_rms = 10.0 ** (target_lufs_rms / 20.0)
    audio = audio * (target_rms / rms)
    max_peak = np.max(np.abs(audio))
    if max_peak > 0.89: audio = audio * (0.89 / max_peak)
    return audio.astype(np.float32)

# Scan & Prepare Clean Reference Audio
all_audio_files = sorted(glob.glob(f"{WORK_DIR}/*.mp3") + glob.glob(f"{WORK_DIR}/*.wav") + glob.glob("/content/*.mp3") + glob.glob("/content/*.wav"))
female_sample, male_sample = None, None
for fpath in all_audio_files:
    fname = os.path.basename(fpath).lower()
    try:
        y, sr = librosa.load(fpath, sr=16000, mono=True)
        y_m = studio_voice_mastering(y, sr=16000, target_lufs_rms=-18.0)
        if any(k in fname for k in ["401087", "403328", "405750", "female", "girl", "woman", "sreymom"]):
            if female_sample is None: female_sample = y_m[:int(3.8 * 16000)]
        elif any(k in fname for k in ["401095", "405777", "male", "boy", "man", "piseth"]):
            if male_sample is None: male_sample = y_m[:int(3.8 * 16000)]
        else:
            if female_sample is None: female_sample = y_m[:int(3.8 * 16000)]
            elif male_sample is None: male_sample = y_m[:int(3.8 * 16000)]
    except Exception:
        pass

female_master_path = os.path.join(WORK_DIR, "master_female_clean.wav")
male_master_path = os.path.join(WORK_DIR, "master_male_clean.wav")
if female_sample is not None: sf.write(female_master_path, female_sample, 16000)
if male_sample is not None: sf.write(male_master_path, male_sample, 16000)

if not os.path.exists(female_master_path):
    try:
        import edge_tts
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        loop.run_until_complete(edge_tts.Communicate("សួស្តី នេះជាសំឡេងស្រីស្តង់ដារសម្រាប់បកប្រែសម្រាយរឿង។", "km-KH-SreymomNeural").save(female_master_path))
        loop.close()
    except Exception: pass

if not os.path.exists(male_master_path):
    try:
        import edge_tts
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        loop.run_until_complete(edge_tts.Communicate("សួស្តី នេះជាសំឡេងប្រុសស្តង់ដារសម្រាប់បកប្រែសម្រាយរឿង។", "km-KH-PisethNeural").save(male_master_path))
        loop.close()
    except Exception: pass

# 3. Load VoxCPM2 Foundation Model
if torch.cuda.is_available(): torch.cuda.empty_cache()
print("\n🧠 3/4 Loading VoxCPM2 Foundation Model (2B Parameters)...")
from voxcpm import VoxCPM
voxcpm_model = VoxCPM.from_pretrained("openbmb/VoxCPM2", optimize=False, load_denoiser=False)
sample_rate = getattr(voxcpm_model.tts_model, "sample_rate", 48000)
print(f"🎉 Official VoxCPM2 Ready! Native 48kHz Studio Quality.")

VOICE_PRESETS = {
    "female_sweet": {"id": "female_sweet", "name": "តួស្រីសម្រាយរឿង (Female Story Narrator)", "gender": "female", "seed": 200302},
    "male_hero": {"id": "male_hero", "name": "តួប្រុសសម្រាយរឿង (Male Story Narrator)", "gender": "male", "seed": 100201},
    "female_lively": {"id": "female_lively", "name": "នារីរស់រវើក/កំប្លែង (Lively Female)", "gender": "female", "seed": 200505},
    "kid_girl": {"id": "kid_girl", "name": "ក្មេងស្រី (Cute Girl)", "gender": "female", "seed": 400504},
    "kid_boy": {"id": "kid_boy", "name": "ក្មេងប្រុស (Playful Boy)", "gender": "male", "seed": 300403},
    "elder_male": {"id": "elder_male", "name": "លោកតា (Wise Grandfather)", "gender": "male", "seed": 500605},
    "elder_female": {"id": "elder_female", "name": "លោកយាយ (Kind Grandmother)", "gender": "female", "seed": 600706},
    "villain": {"id": "villain", "name": "តួកាច / មេបិសាច (Action Villain)", "gender": "male", "seed": 700807},
    "news_host": {"id": "news_host", "name": "ពិធីករ / ព័ត៌មាន (News Anchor)", "gender": "male", "seed": 800908}
}

def clean_khmer_text_for_voxcpm(raw_text: str) -> str:
    if not raw_text: return ""
    t = re.sub(r'Orig\s*:\s*["\'].*?["\']', '', raw_text, flags=re.IGNORECASE)
    t = re.sub(r'\(.*?\)|\[.*?\]', '', t)
    t = re.sub(r'^(តួប្រុស|តួស្រី|អ្នកសម្រាយ|អ្នកសម្រាយរឿង|តាចាស់|យាយចាស់|កុមារ|កូនក្មេង|មេក្រុម|មេបញ្ជាការ|[^\s:៖]{2,15})\s*[:៖-]\s*', '', t)
    trans_map = {r'\bMarcus\b': 'ម៉ាកុស', r'\bElena\b': 'អេលេណា', r'\bSWAT\b': 'ស្វាត', r'\bCyber\b': 'សាយប័រ', r'\bVault\b': 'វ៉ូល', r'\bPolice\b': 'ប៉ូលីស', r'\bHeist\b': 'ហាយស៍', r'\bFlash\b': 'ហ្វ្លាស', r'\bLaser\b': 'ឡាស៊ែរ', r'\bTeam\b': 'ក្រុម', r'\bMonaco\b': 'ម៉ូណាកូ'}
    for pat, repl in trans_map.items(): t = re.sub(pat, repl, t, flags=re.IGNORECASE)
    t = re.sub(r'[a-zA-Z\u4e00-\u9fa5]+', ' ', t)
    t = re.sub(r'[\r\n\t]+', ' ', t)
    t = re.sub(r'\s+', ' ', t).strip()
    return t or raw_text.strip()

app = FastAPI(title="Official VoxCPM2 Server")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])

class CloneRequest(BaseModel):
    text: str
    audio_base64: str = ""
    preset_id: str = ""
    speed: float = 1.0
    gender: str = "female"

@app.get("/")
@app.get("/api/health")
def health():
    return {"status": "online", "gpu": gpu_name, "model": "Official VoxCPM2 (Turbo 48kHz)", "sample_rate": sample_rate}

@app.get("/api/presets")
def presets():
    return {"success": True, "presets": list(VOICE_PRESETS.values())}

@app.post("/api/clone")
@app.post("/api/tts")
def clone_voice(req: CloneRequest):
    if not req.text.strip(): raise HTTPException(status_code=400, detail="Text is required")
    clean_text = clean_khmer_text_for_voxcpm(req.text)
    is_female = (req.gender == "female" or "female" in str(req.preset_id).lower() or "girl" in str(req.preset_id).lower())
    target_preset_id = req.preset_id.strip() if req.preset_id else ("female_sweet" if is_female else "male_hero")
    with tempfile.TemporaryDirectory() as tmpdir:
        ref_audio_path = os.path.join(tmpdir, "ref_sample.wav")
        out_audio_path = os.path.join(tmpdir, "out_cloned.wav")
        if req.audio_base64 and not req.audio_base64.startswith("preset:"):
            ref_bytes = base64.b64decode(req.audio_base64)
            with open(ref_audio_path, "wb") as f: f.write(ref_bytes)
            try:
                y_c, sr_c = librosa.load(ref_audio_path, sr=16000, mono=True)
                y_m = studio_voice_mastering(y_c, sr=16000, target_lufs_rms=-18.0)
                sf.write(ref_audio_path, y_m[:int(3.8 * 16000)], 16000)
            except Exception: pass
            target_ref = ref_audio_path
            used_engine = "VoxCPM2 (Zero-Shot Cloned Voice)"
            target_preset_id = "custom"
        else:
            if target_preset_id not in VOICE_PRESETS: target_preset_id = "female_sweet" if is_female else "male_hero"
            preset = VOICE_PRESETS[target_preset_id]
            used_engine = f"VoxCPM2 Studio ({preset['name']})"
            target_ref = female_master_path if is_female and os.path.exists(female_master_path) else (male_master_path if os.path.exists(male_master_path) else female_master_path)

        steps = 8 if len(clean_text) < 40 else 10
        try:
            with torch.inference_mode():
                call_kwargs = {"text": clean_text, "cfg_value": 1.5, "inference_timesteps": steps}
                if target_ref and os.path.exists(target_ref): call_kwargs["prompt_wav_path"] = target_ref
                try:
                    wav = voxcpm_model.generate(**call_kwargs)
                except TypeError:
                    if "prompt_wav_path" in call_kwargs:
                        call_kwargs["reference_wav_path"] = call_kwargs.pop("prompt_wav_path")
                        try: wav = voxcpm_model.generate(**call_kwargs)
                        except TypeError:
                            call_kwargs.pop("reference_wav_path", None)
                            wav = voxcpm_model.generate(**call_kwargs)
                    else: wav = voxcpm_model.generate(text=clean_text, cfg_value=1.5, inference_timesteps=steps)
            
            pitch_shift = 0.0
            if target_preset_id == 'kid_girl': pitch_shift = 3.5
            elif target_preset_id == 'kid_boy': pitch_shift = 2.8
            elif target_preset_id == 'elder_male': pitch_shift = -3.0
            elif target_preset_id == 'elder_female': pitch_shift = -2.0
            elif target_preset_id == 'villain': pitch_shift = -3.8
            mastered_wav = studio_voice_mastering(wav, sr=sample_rate, target_lufs_rms=-18.0, pitch_shift_semitones=pitch_shift)
            sf.write(out_audio_path, mastered_wav, sample_rate, format='WAV')
        except Exception as e:
            import edge_tts
            voice = "km-KH-SreymomNeural" if is_female else "km-KH-PisethNeural"
            speed_str = f"{int((req.speed - 1.0) * 100):+d}%" if req.speed != 1.0 else "+0%"
            loop = asyncio.new_event_loop()
            asyncio.set_event_loop(loop)
            loop.run_until_complete(edge_tts.Communicate(clean_text, voice, rate=speed_str).save(out_audio_path))
            loop.close()
            used_engine = f"EdgeTTS Studio ({voice})"

        with open(out_audio_path, "rb") as f: out_bytes = f.read()
        return {"success": True, "audio_base64": base64.b64encode(out_bytes).decode("utf-8"), "engine": used_engine}

def get_free_port(start_port=8000):
    for p in range(start_port, start_port + 50):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            if s.connect_ex(('127.0.0.1', p)) != 0: return p
    return start_port

SERVER_PORT = get_free_port(8000)

if try_cloudflare:
    tunnel = try_cloudflare(port=SERVER_PORT)
    tunnel_url = getattr(tunnel, "tunnel", str(tunnel))
    print("\n" + "═" * 80)
    print("🎉 OFFICIAL VOXCPM2 TURBO STUDIO SERVER IS LIVE & READY!")
    print(f"👉 Public API URL: {tunnel_url}")
    print(f"👉 Local API URL:  http://127.0.0.1:{SERVER_PORT}")
    print("═" * 80)
    print("\n📋 ចម្លងយក Public API URL ខាងលើ ទៅបិទភ្ជាប់ក្នុង BT-Dubber រួចចុច 'ផ្ទៀងផ្ទាត់'!")
else:
    print(f"👉 Server running on http://127.0.0.1:{SERVER_PORT}")

def run_uvicorn_in_thread():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    config = uvicorn.Config(app, host="0.0.0.0", port=SERVER_PORT, log_level="info")
    server = uvicorn.Server(config)
    loop.run_until_complete(server.serve())

print("🚀 Starting FastAPI Server in isolated thread...")
server_thread = threading.Thread(target=run_uvicorn_in_thread, daemon=True)
server_thread.start()
time.sleep(2.0)

if __name__ == "__main__":
    try:
        while True:
            time.sleep(1.0)
    except (KeyboardInterrupt, SystemExit):
        print("\n🛑 Server stopped.")
